# Module 4.5: ExoMiner++ Vetting

**NASA Attribution:**
This module uses ExoMiner++ algorithms from NASA's GitHub repository (https://github.com/nasa/ExoMiner). 
ExoMiner++ is a deep learning-based system developed by NASA Ames Research Center for automated vetting of exoplanet candidates from TESS data.

**Reference:**
- ExoMiner++: Enhanced Transit Classification and a New Vetting Catalog for 2-minute TESS Data, The Astronomical Journal
- ExoMiner with Multiplicity Boost of Transit Signal Classifiers: Validation of 69 New Exoplanets using the Multiplicity Boost of ExoMiner, Astronomical Journal, Volume 166, Number 1 (2023)

**Purpose:**
This module applies NASA's ExoMiner++ deep learning models to vet transit candidates detected in Module 4, filtering out false positives and identifying high-confidence planet candidates.

In [ ]:
import streamlit as st
import pandas as pd
import sys
import os
import subprocess
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Add src to path
sys.path.append(os.path.join(os.path.dirname(''), '..', 'src'))

st.set_page_config(
    page_title="Module 4.5: ExoMiner++ Vetting",
    page_icon="🤖",
    layout="wide",
    initial_sidebar_state="collapsed",
)

In [ ]:
# Load CSS
def local_css(file_name):
    with open(file_name) as f:
        st.markdown(f'<style>{f.read()}</style>', unsafe_allow_html=True)

local_css("streamlit_app/static/style.css")

In [ ]:
# Page header
st.markdown("""
<div style="text-align: center; margin-top: 0.5rem; margin-bottom: 1rem;">
    <div style="font-size: 3rem; line-height: 1;">🤖</div>
    <h1 style="margin: 0.25rem 0 0 0; font-weight: 700; font-size: 1.5rem;">
        Module 4.5: ExoMiner++ Vetting
    </h1>
</div>
""", unsafe_allow_html=True)

st.markdown("---")

# NASA Attribution Box
st.markdown("""
<div style="background-color: #e3f2fd; border-left: 4px solid #2196f3; padding: 12px; margin: 16px 0; border-radius: 4px;">
    <strong>🔬 NASA Attribution:</strong> This module uses ExoMiner++ algorithms from NASA's GitHub repository 
    (<a href="https://github.com/nasa/ExoMiner" target="_blank">github.com/nasa/ExoMiner</a>). 
    ExoMiner++ is a deep learning-based system developed by NASA Ames Research Center for automated vetting of exoplanet candidates.
</div>
""", unsafe_allow_html=True)

In [ ]:
# Import module logic
from modules.integrity_tracker import IntegrityTracker

from workspace import current_user, get_store, normalize_user_id

In [ ]:
# Import module logic
from modules.integrity_tracker import IntegrityTracker

In [ ]:
# ExoMiner++ Setup Check
st.markdown("### ExoMiner++ Setup")

exominer_dir = os.path.join(os.path.dirname(''), '..', 'external', 'ExoMiner')

if os.path.exists(exominer_dir):
    st.success("✅ ExoMiner++ repository found")
    st.caption(f"Location: {exominer_dir}")
else:
    st.warning("⚠️ ExoMiner++ repository not found")
    st.caption("You need to clone/download ExoMiner++ from NASA's GitHub repository")
    
    with st.expander("Setup Instructions"):
        st.markdown("""
        **To set up ExoMiner++:**
        
        1. Clone the repository:
           ```bash
           cd external
           git clone https://github.com/nasa/ExoMiner.git
           ```
        
        2. Install dependencies (see ExoMiner README for details)
        
        3. Download pre-trained models (or train your own)
        
        **Alternative:** Use Podman image for seamless setup (see ExoMiner documentation)
        """)

In [ ]:
# Load data from workspace
st.markdown("---")
st.markdown("### Load Data")

user = current_user()
if user:
    store = get_store()
    uid = normalize_user_id(user)
    
    workspace_dir = os.path.join("data", "users", uid)
    
    if os.path.exists(workspace_dir):
        saved_files = [f for f in os.listdir(workspace_dir) if f.endswith('.csv')]
        
        if saved_files:
            selected_file = st.selectbox("Select saved dataset from Module 4 (Transit Detection)", saved_files)
            
            if st.button("📂 Load Dataset"):
                file_path = os.path.join(workspace_dir, selected_file)
                df = pd.read_csv(file_path)
                st.success(f"Loaded {len(df)} coordinates from {selected_file}")
                st.dataframe(df.head(10), use_container_width=True)
                st.session_state['loaded_data'] = df
        else:
            st.caption("No saved datasets found. Complete Module 4 first.")
    else:
        st.caption("No saved datasets found. Complete Module 4 first.")
else:
    st.caption("Sign in to load your saved datasets.")

In [ ]:
# Integrity check
if 'loaded_data' in st.session_state:
    df = st.session_state['loaded_data']
    
    tracker = IntegrityTracker()
    # Check if Module 4 is complete
    passed, failed_count, failed_ids = tracker.check_prerequisite(df, 4)
    
    if not passed:
        st.error(f"⚠️ **Integrity Check Failed**\n\n{failed_count} coordinates have not passed Module 4 (Transit Detection). Please run Module 4 first.")
        st.stop()

In [ ]:
# ExoMiner++ Configuration
if 'loaded_data' in st.session_state:
    st.markdown("---")
    st.markdown("### Vetting Configuration")
    
    vetting_threshold = st.slider(
        "ExoMiner++ vetting threshold",
        min_value=0.0,
        max_value=1.0,
        value=0.5,
        step=0.1,
        help="Candidates with ExoMiner++ score above this threshold are considered vetted"
    )
    
    use_mock = st.checkbox("Use mock vetting (for testing)", value=True, help="Simulate ExoMiner++ vetting without running actual models")

In [ ]:
# Run ExoMiner++ Vetting
if 'loaded_data' in st.session_state:
    st.markdown("---")
    
    col1, col2 = st.columns([1, 3])
    with col1:
        run_vetting = st.button("▶️ Run ExoMiner++ Vetting", type="primary")
    with col2:
        st.caption("Apply NASA's deep learning models to vet transit candidates")
    
    if run_vetting:
        df = st.session_state['loaded_data'].copy()
        
        if use_mock:
            # Mock vetting for testing
            st.info("🧪 Using mock vetting for testing...")
            
            # Simulate ExoMiner++ scores
            import numpy as np
            np.random.seed(42)
            df['exominer_score'] = np.random.uniform(0, 1, len(df))
            df['exominer_vetted'] = df['exominer_score'] >= vetting_threshold
            
            vetted_count = int(df['exominer_vetted'].sum())
            rejected_count = len(df) - vetted_count
            
            st.success(f"✅ Vetting complete: {vetted_count} candidates vetted, {rejected_count} rejected")
        else:
            # TODO: Implement actual ExoMiner++ integration
            st.warning("⚠️ Actual ExoMiner++ integration not yet implemented")
            st.caption("Please set up ExoMiner++ repository and models first")
            
            # Placeholder for actual ExoMiner++ call
            # This would involve:
            # 1. Preparing light curves in ExoMiner++ format
            # 2. Running ExoMiner++ preprocessing
            # 3. Running inference with trained models
            # 4. Parsing results
            
            # For now, use mock
            import numpy as np
            np.random.seed(42)
            df['exominer_score'] = np.random.uniform(0, 1, len(df))
            df['exominer_vetted'] = df['exominer_score'] >= vetting_threshold
            
            vetted_count = int(df['exominer_vetted'].sum())
            rejected_count = len(df) - vetted_count
        
        # Mark as Module 4.5 complete
        df = tracker.mark_module_complete(df, 4.5)
        
        # Display results
        st.subheader("Vetting Results")
        
        col1, col2, col3 = st.columns(3)
        col1.metric("📊 Total Candidates", len(df))
        col2.metric("✅ Vetted", vetted_count)
        col3.metric("❌ Rejected", rejected_count)
        
        # Show vetted vs rejected
        st.markdown("#### Vetted Candidates (passed ExoMiner++ threshold)")
        vetted_df = df[df['exominer_vetted'] == True]
        st.dataframe(vetted_df.head(10), use_container_width=True)
        
        st.markdown("#### Rejected Candidates (below ExoMiner++ threshold)")
        rejected_df = df[df['exominer_vetted'] == False]
        st.dataframe(rejected_df.head(10), use_container_width=True)
        
        # Filter to only vetted candidates for next module
        vetted_only = st.checkbox("Pass only vetted candidates to next module", value=True)
        
        if vetted_only:
            df = df[df['exominer_vetted'] == True].copy()
            st.info(f"Filtering to {len(df)} vetted candidates for Module 5")
        
        st.session_state['module4_5_results'] = df

In [ ]:
# Save to workspace
if 'module4_5_results' in st.session_state:
    st.markdown("---")
    st.markdown("### Save to Workspace")
    
    if user:
        dataset_name = st.text_input(
            "Dataset name",
            value=f"module4_5_exominer_vetted_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}"
        )
        
        if st.button("💾 Save to Workspace"):
            try:
                file_path = os.path.join(workspace_dir, f"{dataset_name}.csv")
                st.session_state['module4_5_results'].to_csv(file_path, index=False)
                st.success(f"Saved {len(st.session_state['module4_5_results'])} candidates as '{dataset_name}.csv'")
            except Exception as exc:
                st.error(f"Failed to save: {exc}")

In [ ]:
# Navigation
st.markdown("---")
st.markdown("### Next Steps")

st.caption("Vetted candidates will proceed to Module 5: Habitability Scoring")

col1, col2 = st.columns(2)
with col1:
    if st.button("💧 Go to Module 5"):
        st.info("Opening Module 5: Habitability Scoring...")
with col2:
    if st.button("🏠 Return to Navigation Hub"):
        st.info("Returning to Navigation Hub...")